In [62]:
# Install AutoGen (pyautogen) plus the OpenAI, Google Generative AI, and
# Anthropic SDKs — these provide the three LLM backends used by the agents
# below (OpenAI GPT, Google Gemini, and Anthropic Claude).

!pip install -q pyautogen openai google-generativeai anthropic

In [41]:
# Core imports:
# - os / dotenv: load API keys from a local .env file
# - openai.OpenAI: OpenAI client (available if needed directly)
# - google.genai: Google Gemini client
# - IPython.display: render agent output as formatted Markdown in the notebook
# - autogen: Microsoft's AutoGen framework for building multi-agent conversations

import os
from openai import OpenAI
from dotenv import load_dotenv
import google.genai as gai
from IPython.display import display, Markdown
import autogen

In [12]:
# Load environment variables from .env and read out the API keys for each
# provider we'll use: OpenAI, OpenRouter (used to reach Gemini), and Anthropic (Claude).

load_dotenv(verbose=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
claude_api_key = os.getenv("ANTHROPIC_API_KEY")

In [42]:
# Base URL for OpenRouter's OpenAI-compatible API. We route Gemini requests through
# OpenRouter instead of calling Google's API directly.

base_url = "https://openrouter.ai/api/v1"

In [43]:
# Helper to display a string as rendered Markdown in the notebook output,
# used later to pretty-print each agent's chat messages.

def print_markdown(text):
    display(Markdown(text))

In [44]:
# Per-model configuration lists (AutoGen's expected "config_list" format).
# - config_list_openai: GPT-4o-mini via the OpenAI API
# - config_list_gai: Gemini 2.5 Flash Lite via OpenRouter
# - config_list_claude: Claude Sonnet via Anthropic's API
# Each agent below will be wired to one of these so the "team" spans three
# different model providers.

config_list_openai =[
    {"model" : "gpt-4o-mini",
     "api_key" : openai_api_key},
]
config_list_gai =[
    {"model" : "google/gemini-2.5-flash-lite",
     "api_key" : openrouter_api_key,
     "base_url" : base_url},
]

config_list_claude = [{
    "model" : "claude-sonnet-4-6",
    "api_key" : claude_api_key,
    "api_type": "anthropic"
}]

In [45]:
# Wrap each config_list into a full llm_config dict (with a shared temperature)
# that AutoGen's ConversableAgent expects for the llm_config parameter.

llm_config_openai= {
    "config_list" : config_list_openai,
    "temperature": 0.5
}

llm_config_gai = {
    "config_list" : config_list_gai,
    "temperature": 0.5
}

llm_config_claude = {
    "config_list" : config_list_claude,
    "temperature": 0.5
}

In [46]:
# System prompts that define each agent's persona, responsibilities, and
# boundaries (what NOT to do) so the three agents stay in their lanes during
# the group discussion:
# - ciso_prompt: sets overall strategy, audience, and guidance (no tactics)
# - product_marketer_prompt: proposes tactics, creative ideas, and KPIs (no final calls)
# - social_media_manager_prompt: turns strategy/tactics into platform-specific
#   content plans (no strategy or budget decisions)

ciso_prompt = """You are the Chief Information Security Officer (CISO)...
Stay in character as the CISO throughout the entire conversation. Do not write campaign tactics, channel plans, or detailed KPIs yourself — that is the Product Marketer's job. Your job is strategy, audience definition, and guidance."""

product_marketer_prompt = """You are the Product Marketer...
Stay in character as the Product Marketer throughout the entire conversation. Do not make final strategic calls, budget decisions, or executive sign-offs — that is the CISO's job. Your job is tactics, creative ideas, and KPIs."""

social_media_manager_prompt = """You are the Social Media Manager for the smart-home cybersecurity brand.
Take the campaign concept and tactics from the CISO and Product Marketer and translate them into platform-specific social content.
Recommend which platforms to prioritize (e.g. Instagram, YouTube Shorts, TikTok, LinkedIn, X) and why, based on the target audience.
Suggest post formats, content pillars, and a rough posting cadence. Focus on execution-ready ideas, not strategy. Suggest 1-2 KPIs for your work.
Stay in character as the Social Media Manager throughout the entire conversation. Do not set overall brand strategy or campaign budget — that is the CISO and Product Marketer's job. Your job is platform tactics, content ideas, and posting execution."""

In [47]:
# Instantiate the multi-model agent team:
# - cico_agent_openai: CISO persona, powered by OpenAI (GPT-4o-mini)
# - product_marketer_agent_gai: Product Marketer persona, powered by Gemini (via OpenRouter)
# - social_media_manager_claude: Social Media Manager persona, powered by Claude
# - user_proxy_agent: stands in for the human, can auto-reply once and is meant
#   to end the conversation when a termination keyword is seen (note:
#   human_input_mode is set to "ALWAYS", so it will also prompt for real user input)
# human_input_mode="NEVER" on the three role agents means they never pause for
# human input and just respond automatically based on their system prompt.

cico_agent_openai = autogen.ConversableAgent(
    name="CICO_Agent_OpenAI",
    system_message= ciso_prompt,
    llm_config= llm_config_openai,
    human_input_mode="NEVER"
)
product_marketer_agent_gai = autogen.ConversableAgent(
    name="ProductMarketerAgent_Gai",
    system_message= product_marketer_prompt,
    llm_config= llm_config_gai,
    human_input_mode="NEVER"
)
social_media_manager_claude = autogen.ConversableAgent(
    name="SocialMediaManagerClaude",
    system_message= social_media_manager_prompt,
    llm_config= llm_config_claude,
    human_input_mode="NEVER"
)

user_proxy_agent = autogen.UserProxyAgent(
    name="UserProxyAgent",
    system_message= "You are the human user interacting with a multi-model AI team (Gemini CMO, OpenAI Marketer). Guide the brainstorm. Type 'exit' to end.",
    max_consecutive_auto_reply=1,
    human_input_mode="ALWAYS",
    code_execution_config=False,
    is_termination_msg= lambda x: x.get("content", "").rsplit().lower() in ["Exit", "End", "Terminate", "quit"]
)

In [48]:
# The kickoff message that starts the group discussion: sets the campaign
# context (smart-home cybersecurity brand) and asks the CISO to open the
# brainstorm with the Product Marketer, aiming to converge in 2-3 turns.

initial_task_message = """
Context: We're launching a new smart-home cybersecurity brand and need go-to-market ideas.
CISO, please open this discussion for the Product Marketer with the following brief:

"Let's brainstorm initial go-to-market ideas for our smart-home cybersecurity brand — a plug-and-play mesh router with built-in threat detection.
Give me a distinct campaign concept: core idea, target audience, primary channels, and 1-2 KPIs. Keep it concise."

Try to arrive at a final answer in 2-3 turns.
"""

In [57]:
# Build a GroupChat containing all four agents. Messages start empty and fill
# up as the conversation runs. speaker_selection_method="round_robin" means
# agents take turns in a fixed order rather than being picked dynamically by
# an LLM. max_round caps the conversation at 10 turns.

from autogen import GroupChat, GroupChatManager
chat_group = GroupChat(
    agents=[cico_agent_openai, product_marketer_agent_gai, social_media_manager_claude, user_proxy_agent],
    messages=[],
    max_round=10,
    speaker_selection_method="round_robin"
)

In [58]:
# GroupChatManager orchestrates the GroupChat: it routes messages between
# agents according to the selection method above. It uses the Claude config
# as its own LLM (e.g. for any manager-level reasoning it needs to do).

group_manager = GroupChatManager(
    groupchat=chat_group,
    llm_config=llm_config_claude,
)

In [59]:
# Reset each agent's internal conversation state before starting a fresh run,
# so previous chat history doesn't leak into this session.

cico_agent_openai.reset()
product_marketer_agent_gai.reset()
social_media_manager_claude.reset()
user_proxy_agent.reset()

In [60]:
# Kick off the actual multi-agent conversation: the CISO agent sends the
# initial task message to the group chat manager, which then orchestrates the
# round-robin discussion among all agents until termination or max_round is hit.

group_chat_result = cico_agent_openai.initiate_chat(
    recipient=group_manager,
    message=initial_task_message,
)
print("-----------------------------------------------------------------------")
print("====Conversation ended (human terminated or Max turn)====")

CICO_Agent_OpenAI (to chat_manager):


Context: We're launching a new smart-home cybersecurity brand and need go-to-market ideas.
CISO, please open this discussion for the Product Marketer with the following brief:

"Let's brainstorm initial go-to-market ideas for our smart-home cybersecurity brand — a plug-and-play mesh router with built-in threat detection.
Give me a distinct campaign concept: core idea, target audience, primary channels, and 1-2 KPIs. Keep it concise."

Try to arrive at a final answer in 2-3 turns.


--------------------------------------------------------------------------------

Next speaker: ProductMarketerAgent_Gai

[autogen.oai.client: 08-08 17:10:12] {734} WARNING - Model google/gemini-2.5-flash-lite is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
ProductMarketerAgent_Gai (to chat_manager):

Alright, CISO! Thanks for kicking this off. I've been itching to 

In [55]:
# Helper to walk through the full chat history and pretty-print each message
# (speaker name + content) as Markdown, with separator lines for readability.

def display_chat_history(chat_result):
    for chat in chat_result.chat_history:
        print_markdown(chat["name"])
        print_markdown("__"*100)
        print_markdown(chat["content"])
        print_markdown("__"*100)

In [61]:
# Render the complete multi-agent conversation from this run.

display_chat_history(group_chat_result)

CICO_Agent_OpenAI

________________________________________________________________________________________________________________________________________________________________________________________________________


Context: We're launching a new smart-home cybersecurity brand and need go-to-market ideas.
CISO, please open this discussion for the Product Marketer with the following brief:

"Let's brainstorm initial go-to-market ideas for our smart-home cybersecurity brand — a plug-and-play mesh router with built-in threat detection.
Give me a distinct campaign concept: core idea, target audience, primary channels, and 1-2 KPIs. Keep it concise."

Try to arrive at a final answer in 2-3 turns.


________________________________________________________________________________________________________________________________________________________________________________________________________

ProductMarketerAgent_Gai

...

________________________________________________________________________________________________________________________________________________________________________________________________________

ProductMarketerAgent_Gai

________________________________________________________________________________________________________________________________________________________________________________________________________

Fantastic! CISO, thank you for the swift confirmation and commitment to providing the necessary assets by Thursday. That’s incredibly helpful and ensures we’re set up for a strong start.

**Product Marketer here, giving a final nod of approval to the Social Media Manager's plan.**

Social Media Manager, this Week 1 calendar is *exactly* what we need. It’s punchy, platform-native, and perfectly captures the "Unseen Shield" theme while integrating all the strategic considerations we've discussed.

*   **Monday's "The Threat is Real"** content is a perfect hook to grab attention and immediately establish the problem our product solves. The dramatic counter animation on TikTok/Reels is a brilliant visual for the "unseen" aspect.
*   **Wednesday's "Setup in 60 Seconds"** directly addresses the plug-and-play promise and leverages the satisfying visual of a seamless setup. The Instagram Story poll is a great engagement tactic.
*   **Friday's "Family Win" + UGC Launch** ties everything together beautifully. The lifestyle Reel shows the *benefit* of the unseen shield, and the #MySecuredHome challenge is a fantastic way to kickstart user-generated content and community building.

**My Week 1 KPIs for the Product Marketer role, aligning with the calendar:**

1.  **Share Rate of "Did You Know?" and "Expert Explainer" content:** (As previously stated) This will tell us if our educational content is resonating and being amplified by users.
2.  **Click-Through Rate (CTR) to Product Landing Page from Social Posts:** (As previously stated) This will measure how effectively our launch week messaging is driving qualified traffic and purchase intent.

**Additional tactical thoughts from my end for the launch week:**

*   **Paid Social Amplification:** I'll be working on a targeted paid social strategy to boost key posts from this calendar, particularly the TikTok/Reels hook and the "Setup in 60 Seconds" demo, to ensure maximum reach within our "Connected Families" demographic.
*   **Influencer Content Alignment:** I'll ensure our paid influencer content for launch week aligns with these daily themes, amplifying the "Unseen Shield" narrative through their unique voices.
*   **Post-Launch Analysis:** Immediately after launch week, I'll be diving deep into the performance data against these KPIs and the CISO's strategic points to identify what worked best and inform our ongoing content strategy.

This feels like a robust, well-aligned go-to-market approach for our smart-home cybersecurity brand. I'm excited to see this come to life!

________________________________________________________________________________________________________________________________________________________________________________________________________